In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications
import os

# 1. SETUP THE DATA LOADER FOR THE NEW FOLDER
# Pointing to your /kaggle/working/ directory
data_dir = '/kaggle/input/et-cropped-dataset-second'

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2, # Subject-independent logic is handled by your folder naming
    subset="training",
    seed=123,
    image_size=(224, 224), # Shifted to 224 for MobileNetV2 efficiency
    batch_size=32,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(224, 224),
    batch_size=32,
    label_mode='categorical'
)

# 2. DEFINE MODEL ARCHITECTURE WITHIN STRATEGY
strategy = tf.distribute.get_strategy()

with strategy.scope():
    # Softened augmentation to keep the iris centered
    data_augmentation = tf.keras.Sequential([
        layers.RandomRotation(0.02), # Very small rotation
        layers.RandomZoom(0.05),     # Minimal zoom to prevent clipping
        layers.RandomBrightness(0.1), # Helps with the blurry samples
    ])

    # MobileNetV2 optimized for 224x224
    base_model = applications.MobileNetV2(
        input_shape=(224, 224, 3), 
        include_top=False, 
        weights='imagenet'
    )
    base_model.trainable = False 
    
    model2 = models.Sequential([
        layers.Input(shape=(224, 224, 3)),
        data_augmentation,
        base_model,
        layers.GlobalAveragePooling2D(),
        
        # Increased Dropout to 0.6 to prevent subject memorization
        layers.Dense(256, kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.6), 
        
        layers.Dense(4, activation='softmax')
    ])

    # Lowered initial learning rate for blurry medical data
    model2.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), 
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), 
        metrics=['accuracy']
    )

# 3. TRAINING EXECUTION
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, verbose=1)
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)

# Custom weights for your 1000-per-class balanced set
custom_weights = {0: 1.0, 1: 1.5, 2: 1.0, 3: 1.5} 

history = model2.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=1, 
    class_weight=custom_weights, 
    callbacks=[reduce_lr, early_stop]
)

Found 4000 files belonging to 4 classes.
Using 3200 files for training.


I0000 00:00:1768498861.298298      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1768498861.302252      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 4000 files belonging to 4 classes.
Using 800 files for validation.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


I0000 00:00:1768498874.201436     132 cuda_dnn.cc:529] Loaded cuDNN version 91002


100/100 ━━━━━━━━━━━━━━━━━━━━ 17s 77ms/step - accuracy: 0.3057 - loss: 6.4472 - val_accuracy: 0.4150 - val_loss: 5.1327 - learning_rate: 1.0000e-04


In [3]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications

# 1. FIX: DEFINE THE STRATEGY
# This tells the code how to use your Kaggle GPUs.
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print("Running on TPU")
except ValueError:
    strategy = tf.distribute.get_strategy() # Default for CPU/GPU
    print(f"Running on {len(tf.config.list_physical_devices('GPU'))} GPUs")

# 2. DEFINE THE HEAD SECTION WITHIN SCOPE
with strategy.scope():
    # Keep the same augmentation used previously
    data_augmentation = tf.keras.Sequential([
        layers.RandomRotation(0.02),
        layers.RandomZoom(0.05),
        layers.RandomBrightness(0.1),
    ])

    base_model = applications.MobileNetV2(
        input_shape=(224, 224, 3), 
        include_top=False, 
        weights='imagenet'
    )
    base_model.trainable = False # Locked for stabilization
    
    model_gmp = models.Sequential([
        layers.Input(shape=(224, 224, 3)),
        data_augmentation,
        base_model,
        # GLOBAL MAX POOLING: Focuses on the sharpest iris signal
        layers.GlobalMaxPooling2D(), 
        
        layers.Dense(256, kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.6), 
        
        layers.Dense(4, activation='softmax')
    ])

    # 3. OPTIMIZED COMPILATION
    model_gmp.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5), 
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), 
        metrics=['accuracy']
    )

# 4. SURGICAL CLASS WEIGHTS
# Protecting the 0.78 Recall for 'High' severity
custom_weights = {0: 1.5, 1: 1.0, 2: 1.2, 3: 1.0} 

# 5. EXECUTION
print("\n--- STABILIZING PRECISION HEAD (GLOBAL MAX POOLING) ---")
model_gmp.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=1, 
    class_weight=custom_weights,
    callbacks=[reduce_lr, early_stop]
)

Running on 2 GPUs

--- STABILIZING PRECISION HEAD (GLOBAL MAX POOLING) ---
100/100 ━━━━━━━━━━━━━━━━━━━━ 13s 78ms/step - accuracy: 0.2773 - loss: 6.5600 - val_accuracy: 0.3762 - val_loss: 5.5715 - learning_rate: 5.0000e-05


In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications

# 1. SE BLOCK DEFINITION
def se_block(inputs, ratio=16):
    channels = inputs.shape[-1]
    # Squeeze: Captures global spatial information
    se = layers.GlobalAveragePooling2D()(inputs)
    # Excitation: Learns channel-specific importance
    se = layers.Dense(channels // ratio, activation='relu')(se)
    se = layers.Dense(channels, activation='sigmoid')(se)
    # Rescale inputs: Multiplies original features by the learned weights
    se = layers.Reshape((1, 1, channels))(se)
    return layers.Multiply()([inputs, se])

# 2. DEFINE MODEL WITHIN STRATEGY SCOPE
with strategy.scope():
    # Input and Augmentation
    inputs = layers.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)
    
    # MobileNetV2 Base
    base_model = applications.MobileNetV2(
        input_shape=(224, 224, 3), 
        include_top=False, 
        weights='imagenet'
    )
    base_model.trainable = False 
    
    # Connect Base to SE Block
    base_out = base_model(x)
    attn_out = se_block(base_out, ratio=16) 
    
    # Global Max Pooling to kill bleeding
    x = layers.GlobalMaxPooling2D()(attn_out)
    
    # Dense Head Logic
    x = layers.Dense(256, kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.6)(x)
    
    outputs = layers.Dense(4, activation='softmax')(x)

    # Define the final model_gmp
    model_gmp = models.Model(inputs, outputs)

    # 3. OPTIMIZED COMPILATION
    model_gmp.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5), 
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), 
        metrics=['accuracy']
    )

# 4. SURGICAL CLASS WEIGHTS
custom_weights = {0: 1.5, 1: 1.0, 2: 1.2, 3: 1.0} 

# 5. EXECUTION
print("\n--- STABILIZING ATTENTION-AUGMENTED HEAD ---")
model_gmp.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=160, 
    class_weight=custom_weights,
    callbacks=[reduce_lr, early_stop]
)



--- STABILIZING ATTENTION-AUGMENTED HEAD ---
Epoch 1/160
100/100 ━━━━━━━━━━━━━━━━━━━━ 13s 80ms/step - accuracy: 0.2581 - loss: 6.5115 - val_accuracy: 0.3237 - val_loss: 5.5526 - learning_rate: 5.0000e-05
Epoch 2/160
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 64ms/step - accuracy: 0.3115 - loss: 6.0828 - val_accuracy: 0.3862 - val_loss: 5.3229 - learning_rate: 5.0000e-05
Epoch 3/160
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 64ms/step - accuracy: 0.3555 - loss: 5.8760 - val_accuracy: 0.4725 - val_loss: 5.1391 - learning_rate: 5.0000e-05
Epoch 4/160
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 64ms/step - accuracy: 0.3746 - loss: 5.6747 - val_accuracy: 0.4913 - val_loss: 5.0002 - learning_rate: 5.0000e-05
Epoch 5/160
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 64ms/step - accuracy: 0.4068 - loss: 5.4814 - val_accuracy: 0.5138 - val_loss: 4.8696 - learning_rate: 5.0000e-05
Epoch 6/160
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 65ms/step - accuracy: 0.4068 - loss: 5.4041 - val_accuracy: 0.5188 - val_loss: 4.7618 - learning_rate: 5.0000e-05
Epoch

In [5]:
# Unfreeze the top 10 layers in small steps
for i in range(2, 11, 2):
    print(f"\n--- UNFREEZING TOP {i} LAYERS ---")
    with strategy.scope():
        base_model.trainable = True
        # Lock everything except the last i layers
        for layer in base_model.layers[:-i]:
            layer.trainable = False
            
        # Keep BatchNormalization frozen for safety
        for layer in base_model.layers[-i:]:
            if isinstance(layer, layers.BatchNormalization):
                layer.trainable = False

        # Use a VERY small learning rate
        model_gmp.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=2e-6),
            loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
            metrics=['accuracy']
        )

    model_gmp.fit(
        train_ds,
        validation_data=val_ds,
        epochs=15, # Shorter epochs to prevent overfitting
        class_weight=custom_weights,
        callbacks=[early_stop]
    )


--- UNFREEZING TOP 2 LAYERS ---
Epoch 1/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 14s 81ms/step - accuracy: 0.6780 - loss: 2.3351 - val_accuracy: 0.6625 - val_loss: 2.1863
Epoch 2/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 65ms/step - accuracy: 0.6833 - loss: 2.3405 - val_accuracy: 0.6463 - val_loss: 2.1907
Epoch 3/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 65ms/step - accuracy: 0.6706 - loss: 2.3519 - val_accuracy: 0.6450 - val_loss: 2.1895
Epoch 4/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 65ms/step - accuracy: 0.6778 - loss: 2.3171 - val_accuracy: 0.6438 - val_loss: 2.1897
Epoch 5/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 65ms/step - accuracy: 0.6728 - loss: 2.3497 - val_accuracy: 0.6413 - val_loss: 2.1906
Epoch 6/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 65ms/step - accuracy: 0.6575 - loss: 2.3700 - val_accuracy: 0.6413 - val_loss: 2.1879
Epoch 7/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 66ms/step - accuracy: 0.6844 - loss: 2.3177 - val_accuracy: 0.6438 - val_loss: 2.1840
Epoch 8/15
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 66ms/step - accuracy